# Pertemuan 5 — Membersihkan Data, dengan Alasan
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK5:** Mahasiswa mampu menerapkan teknik *data preparation* dan *data manipulation* dengan Python
disertai justifikasi setiap keputusan (C3).

---

> **Dosen masih berhalangan hadir.** Kelas didampingi asisten. Workbook ini dapat Anda selesaikan sendiri.

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–7 sekitar 60 menit, Bagian 8 sekitar 30 menit, Bagian 9–13 sekitar 35 menit, Bagian 14 sekitar 25 menit. |
| **Prasyarat** | Workbook pertemuan 4 sudah selesai, termasuk Bagian 10 (profil kualitas data tim). |
| **Umpan balik** | Sama seperti pekan lalu: sel `cek(...)` menjawab BENAR atau BELUM tanpa menampilkan kuncinya. |

### Yang Anda hasilkan di akhir sesi

**M2 — Data Quality Report (bobot 10%)**, terdiri atas tiga berkas:

1. `reports/data_quality_report.docx` — laporan temuan dan keputusan
2. `notebooks/02_cleaning.ipynb` — notebook pembersihan yang dapat dijalankan ulang
3. `cleaning_log.md` — catatan setiap keputusan beserta alasannya

Pakai `cek_m2.py` untuk memeriksa kelengkapan kiriman sebelum mengumpulkan.

---
## Bagian 0 — Persiapan

In [20]:
import pandas as pd
import numpy as np
import hashlib
import os

from google.colab import files

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

uploaded = files.upload()

FILE = next(
    nama for nama in uploaded
    if nama.endswith('.csv') and 'transaksi_koperasi_2025' in nama
)

mentah = pd.read_csv(FILE)

print('Data mentah:', mentah.shape)
display(mentah.head())

Saving transaksi_koperasi_2025 (1).csv to transaksi_koperasi_2025 (1) (1).csv
Data mentah: (1543, 16)


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,TRX251385,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,TRX251333,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,TRX251278,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,TRX251304,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,TRX251364,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


In [77]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
_K = {
    'P5_baris_bersih': 'b3396bcb59',
    'P5_dibuang': '1a6562590e',
    'P5_kat': 'ef2d127de3',
    'P5_metode': '4b227777d4',
    'P5_anggota': '7c252ab334',
    'P5_total_num': 'b3396bcb59',
    'P5_curiga': '1a6562590e',
    'P5_rating_kosong': '25fc0e7096',
    'P5_bulan': '4b227777d4',
    'P5_langkah': 'e629fa6598'
}

_P = {
    'P5_baris_bersih': 'Jumlah baris setelah pembersihan.',
    'P5_dibuang': 'Jumlah baris mentah dikurangi baris bersih.',
    'P5_kat': 'Jumlah kategori unik.',
    'P5_metode': 'Jumlah metode pembayaran unik.',
    'P5_anggota': 'Jumlah ID anggota unik.',
    'P5_total_num': 'Jumlah total_bayar yang berupa angka.',
    'P5_curiga': 'Jumlah total transaksi yang mencurigakan.',
    'P5_rating_kosong': 'Jumlah rating layanan yang kosong.',
    'P5_bulan': 'Jumlah bulan yang berbeda.',
    'P5_langkah': 'Jumlah langkah di cleaning log.'
}

def _h(x):
    return hashlib.sha256(
        str(x).strip().lower().encode()
    ).hexdigest()[:10]

def cek(tugas, jawaban):
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas)
        return

    if _h(jawaban) == _K[tugas]:
        print(f'BENAR {tugas} = {jawaban}')
    else:
        print(f'BELUM {tugas} = {jawaban}')
        print('Petunjuk:', _P.get(tugas, '-'))

print('Alat pemeriksa siap.')

Alat pemeriksa siap.


---
## Bagian 1 — Pembersihan adalah keputusan analitis

Pekan lalu Anda menemukan belasan masalah. Hari ini setiap masalah itu menuntut sebuah **keputusan**,
dan hampir tidak ada yang punya jawaban tunggal.

Ambil satu contoh: 29 baris memiliki `jumlah` nol atau negatif. Ada empat pilihan, semuanya masuk akal:

| Pilihan | Masuk akal bila | Risikonya |
|---|---|---|
| Dibuang barisnya | Yakin itu kesalahan input | Kehilangan transaksi yang mungkin sah |
| Dijadikan kosong | Belum tahu penyebabnya | Analisis berkurang datanya |
| Diubah menjadi positif | Yakin itu retur yang salah tanda | Mengarang data bila dugaan salah |
| Dibiarkan, tetapi ditandai | Ingin menyerahkan keputusan ke tahap analisis | Harus diingat terus |

> **Yang dinilai bukan pilihan Anda, melainkan apakah Anda punya alasan dan menuliskannya.**
> Dua tim boleh mengambil keputusan berbeda pada data yang sama dan keduanya memperoleh nilai penuh,
> asalkan alasannya jelas. Tim yang membersihkan tanpa alasan tertulis tidak memperoleh nilai penuh
> meskipun kodenya rapi.

### Tiga aturan yang berlaku sepanjang pertemuan ini

1. **Jangan pernah menimpa data mentah.** `data/raw/` tidak pernah diubah. Hasil disimpan di `data/processed/`.
2. **Satu langkah, satu catatan.** Setiap perubahan masuk ke *cleaning log* saat itu juga, bukan di akhir.
3. **Menandai lebih aman daripada menebak.** Kalau Anda tidak tahu nilai yang benar, jangan mengarang.
   Jadikan kosong atau beri kolom penanda.

---
## Bagian 2 — Urutan pembersihan

Urutan bukan selera. Langkah yang dikerjakan terlalu awal akan membuat langkah berikutnya salah.

| Urutan | Langkah | Mengapa harus di sini |
|---|---|---|
| 1 | Perbaiki **tipe data** | Selama uang masih teks, tidak satu pun pemeriksaan angka bisa jalan |
| 2 | Buang **spasi** pada teks | Kalau tidak, penyeragaman kategori akan meleset |
| 3 | Ubah **placeholder** menjadi kosong | Kalau tidak, `-999` ikut terhitung sebagai nilai |
| 4 | **Seragamkan** kategori | Setelah spasi bersih, barulah sinonim terlihat |
| 5 | Tangani **duplikat** | Setelah teks seragam, duplikat nyaris baru bisa dikenali |
| 6 | Tangani **nilai mustahil** | Perlu tipe data yang benar untuk membandingkan |
| 7 | Tangani **nilai kosong** | Paling akhir, karena langkah 3 dan 6 menambah nilai kosong baru |
| 8 | Buat **kolom turunan** | Hanya bermakna di atas data yang sudah bersih |

> **Kesalahan urutan yang paling sering terjadi:** mengisi nilai kosong lebih dulu, lalu baru mengubah
> placeholder menjadi kosong. Hasilnya, `-999` sudah terlanjur ikut dirata-ratakan, dan nilai kosong yang
> baru muncul tidak pernah tertangani.

---
## Bagian 3 — Langkah 1 dan 2: tipe data

In [22]:
d = mentah.copy()

jejak = []

def catat(langkah, terdampak, alasan):
    jejak.append({
        'langkah': langkah,
        'baris_terdampak': int(terdampak),
        'alasan': alasan
    })

print('Data mentah:', mentah.shape)
print('Salinan data:', d.shape)

Data mentah: (1543, 16)
Salinan data: (1543, 16)


In [38]:
berformat = d['total_bayar'].astype(str).str.startswith('Rp')

d['total_bayar'] = pd.to_numeric(
    d['total_bayar'].astype(str)
      .str.replace('Rp', '', regex=False)
      .str.replace('.', '', regex=False)
      .str.strip(),
    errors='coerce'
)

catat(
    'Konversi total_bayar ke angka',
    berformat.sum(),
    'Buang Rp dan pemisah ribuan agar kolom dapat dihitung'
)

print('Gagal dikonversi:', int(d['total_bayar'].isna().sum()))

Gagal dikonversi: 0


> **Perhatikan `errors='coerce'`.** Nilai yang tidak bisa dikonversi menjadi `NaN` alih-alih menghentikan
> program. Itu sengaja: kita ingin tahu berapa banyak yang gagal, bukan berhenti di nilai pertama yang aneh.
> Selalu cetak jumlah kegagalannya. Kalau angkanya besar, berarti asumsi format Anda salah.

In [39]:
t = d['tanggal_transaksi'].astype(str)
iso = t.str.match(r'^\d{4}-')

baru = pd.Series(
    pd.NaT,
    index=t.index,
    dtype='datetime64[ns]'
)

baru[iso] = pd.to_datetime(
    t[iso],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

baru[~iso] = pd.to_datetime(
    t[~iso],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

d['tanggal_transaksi'] = baru

catat(
    'Konversi tanggal per format',
    len(d),
    'Memisahkan format tanggal agar tidak salah membaca tanggal'
)

print('Gagal:', int(d['tanggal_transaksi'].isna().sum()))
print('Rentang:', d['tanggal_transaksi'].min(), 'sampai', d['tanggal_transaksi'].max())
print('Bulan:', sorted(d['tanggal_transaksi'].dt.month.dropna().unique()))

Gagal: 0
Rentang: 2025-08-01 07:09:00 sampai 2025-11-28 16:01:00
Bulan: [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]


> Uji akal sehat dari pekan lalu dijalankan lagi di sini: empat bulan, Agustus sampai November. Benar.

---
## Bagian 4 — Langkah 2 sampai 4: teks, placeholder, kategori

In [40]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [
    c for c in d.columns
    if d[c].dtype == object or str(d[c].dtype) == 'str'
]

n_spasi = 0

for kol in kolom_teks:
    s = d[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    d[kol] = s.str.strip()

catat(
    'Buang spasi awal dan akhir kolom teks',
    n_spasi,
    'Spasi tersembunyi dapat membuat kategori terhitung berbeda'
)

In [28]:
# CONTOH 4.2 — placeholder menjadi kosong
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

n_ph = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    m = s.str.lower().isin(CURIGA)
    n_ph += int(m.sum())
    d.loc[m, kol] = pd.NA
catat('Placeholder teks menjadi kosong', n_ph, 'N/A, -, null, dan teks kosong bukan nilai sungguhan')

n999 = int((d['rating_layanan'] == -999).sum())
d.loc[d['rating_layanan'] == -999, 'rating_layanan'] = np.nan
catat('Placeholder -999 pada rating menjadi kosong', n999,
      'Angka -999 merusak rata-rata tanpa pernah terdeteksi isna()')

In [29]:
# CONTOH 4.3 — penyeragaman kategori, dengan peta keputusan yang eksplisit
# Bagian yang bisa dikerjakan mesin:
d['kategori'] = d['kategori'].astype('string').str.strip().str.title()

# Bagian yang HARUS diputuskan manusia, ditulis sebagai peta agar bisa ditinjau orang lain:
PETA_KATEGORI = {
    'Atk'        : 'ATK',          # penyeragaman penulisan
    'Alat Tulis' : 'ATK',          # sinonim; dipastikan ke pengurus koperasi
    'Minumam'    : 'Minuman',      # salah ketik yang jelas
}
sebelum = d['kategori'].nunique()
d['kategori'] = d['kategori'].replace(PETA_KATEGORI)

catat('Seragamkan kategori', sebelum - d['kategori'].nunique(),
      'Huruf dan spasi oleh kode; sinonim dan salah ketik diputuskan manual lalu dicatat di peta')
print(sorted(d['kategori'].dropna().unique()))

['ATK', 'Jasa', 'Makanan', 'Minuman', 'Pulsa']


> **Mengapa peta ditulis sebagai dictionary, bukan rangkaian `replace` berantai?** Karena peta bisa dibaca
> orang lain, bisa ditinjau, bisa ditambah, dan ikut masuk ke laporan sebagai lampiran. Rangkaian `replace`
> yang tersebar di beberapa sel tidak bisa.
>
> Catatan pada `'Alat Tulis' : 'ATK'` bukan hiasan. Itu bukti bahwa Anda **bertanya**, bukan menebak.

In [30]:
# CONTOH 4.4 — kolom teks lain
d['metode_bayar'] = d['metode_bayar'].astype('string').str.strip().str.title().replace({'Qris': 'QRIS'})
d['kasir']        = d['kasir'].astype('string').str.strip().str.title()

print('metode_bayar:', sorted(d['metode_bayar'].dropna().unique()))
print('kasir       :', sorted(d['kasir'].dropna().unique()))

metode_bayar: ['QRIS', 'Saldo Koperasi', 'Transfer', 'Tunai']
kasir       : ['Bagus', 'Dimas', 'Rina', 'Sari']


### TUGAS 1 — hasil penyeragaman

In [58]:

# TUGAS 1

p5_kat = d['kategori'].nunique(dropna=True)

p5_metode = d['metode_bayar'].nunique(dropna=True)

p5_anggota = d['id_anggota'].astype('string').str.strip().nunique(dropna=True)

cek('P5_kat', p5_kat)
cek('P5_metode', p5_metode)
cek('P5_anggota', p5_anggota)


BENAR P5_kat = 5
BENAR P5_metode = 4
BENAR P5_anggota = 259


---
## Bagian 5 — Langkah 5: duplikat

In [33]:
# CONTOH 5.1 — duplikat persis
n_persis = int(d.duplicated().sum())
d = d.drop_duplicates()
catat('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan')

> Jumlahnya **lebih besar** daripada yang Anda temukan pekan lalu. Sebabnya: setelah teks diseragamkan,
> baris yang tadinya berbeda hanya pada kapitalisasi kini menjadi benar-benar identik.
>
> Ini contoh mengapa urutan penting. Membersihkan teks lebih dulu membuat duplikat nyaris berubah
> menjadi duplikat persis, sehingga bisa ditangani otomatis.

In [34]:
# CONTOH 5.2 — duplikat yang tersisa pada kunci utama
n_id = int(d['id_transaksi'].duplicated(keep=False).sum())
print('Baris dengan id_transaksi berulang:', n_id)

# Aturan penyatuan: simpan baris yang catatan_kasir-nya terisi.
d = (d.sort_values('catatan_kasir', na_position='last')
       .drop_duplicates(subset=['id_transaksi'], keep='first'))
catat('Satukan duplikat menurut id_transaksi', n_id,
      'Satu transaksi hanya boleh satu baris; dipilih baris yang catatannya terisi')

Baris dengan id_transaksi berulang: 15


In [35]:
# CONTOH 5.3 — kunci utama yang kosong
n_kosong = int(d['id_transaksi'].isna().sum())
d = d[d['id_transaksi'].notna()]
catat('Buang baris tanpa id_transaksi', n_kosong,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber')

---
## Bagian 6 — Langkah 6: nilai mustahil

Pohon keputusan yang dipakai di mata kuliah ini:

```
Nilai ini mustahil.
│
├─ Tahu nilai yang benar?            -> perbaiki, lalu catat sumber kebenarannya
├─ Tahu pasti ini kesalahan input?   -> jadikan kosong
├─ Mungkin sah tetapi belum jelas?   -> biarkan, tambahkan kolom penanda
└─ Barisnya tidak bisa dipakai       -> buang, lalu catat berapa banyak
```

Yang **tidak** ada di pohon ini: menebak. Kalau Anda tidak tahu, jangan mengarang nilai.

In [36]:
# CONTOH 6.1 — jumlah nol atau negatif: dijadikan kosong, bukan dibuang
n_j = int((d['jumlah'] <= 0).sum())
d.loc[d['jumlah'] <= 0, 'jumlah'] = np.nan
catat('jumlah <= 0 menjadi kosong', n_j,
      'Belum jelas retur atau salah input; barisnya masih berguna untuk kolom lain')

In [37]:
# CONTOH 6.2 — diskon di luar rentang
buruk = (d['diskon_persen'] < 0) | (d['diskon_persen'] > 100)
n_dk = int(buruk.sum())
d.loc[buruk, 'diskon_persen'] = np.nan
catat('Diskon di luar 0-100 menjadi kosong', n_dk, 'Nilai mustahil; menunggu konfirmasi pengurus')

In [17]:
# CONTOH 6.3 — pelanggaran aturan: DITANDAI, bukan ditimpa
d['total_seharusnya']   = (d['jumlah'] * d['harga_satuan'] * (1 - d['diskon_persen'] / 100)).round()
d['selisih_total']      = d['total_bayar'] - d['total_seharusnya']
d['total_mencurigakan'] = d['selisih_total'].abs() > 1

catat('Tandai total yang tidak cocok', d['total_mencurigakan'].sum(),
      'Ditandai, tidak ditimpa, karena belum diketahui kolom mana yang keliru')

d.loc[d['total_mencurigakan'], ['jumlah','harga_satuan','diskon_persen','total_bayar','total_seharusnya']].head(5)

  Tandai total yang tidak cocok: 50 baris


,jumlah,harga_satuan,diskon_persen,total_bayar,total_seharusnya
1381,4.0,5000,0.0,200000,20000.0
1401,3.0,6000,5.0,8550,17100.0
148,7.0,5000,0.0,350000,35000.0
897,1.0,35000,0.0,350000,35000.0
1483,2.0,6000,0.0,24000,12000.0


> **Mengapa ditandai, bukan diperbaiki?** Karena kita tidak tahu kolom mana yang salah. Bisa `total_bayar`
> yang keliru, bisa `jumlah`, bisa `harga_satuan`. Menimpa `total_bayar` dengan hasil hitungan sama saja
> dengan memutuskan bahwa tiga kolom lain pasti benar — tanpa dasar apa pun.
>
> Kolom penanda memindahkan keputusan ke tahap analisis, di mana dampaknya bisa diuji:
> jalankan analisis dengan dan tanpa baris bertanda, lalu lihat apakah kesimpulannya berubah.

---
## Bagian 7 — Langkah 7: nilai kosong, dan mengapa tidak ada pilihan yang netral

Empat strategi yang lazim dipakai:

| Strategi | Caranya | Cocok untuk |
|---|---|---|
| Buang barisnya | `dropna()` | MCAR, dan hanya bila yang hilang sedikit |
| Isi nilai tengah | `fillna(mean)` atau `fillna(median)` | MCAR, kolom yang bukan fokus analisis |
| Isi per kelompok | `groupby().transform()` | MAR, bila kelompoknya memang penjelasnya |
| Tandai sebagai kategori | `fillna('Tidak tercatat')` | Kolom kategorikal; kekosongan itu sendiri bermakna |

Pekan lalu kita menduga `saldo_simpanan` bersifat **MNAR** — nilai besar lebih sering disembunyikan.
Sekarang kita uji dugaan itu.

### Keistimewaan data latihan

Dataset ini sintetis, dan itu memberi kita satu kemampuan yang **tidak pernah Anda miliki pada data nyata**:
kita bisa melihat nilai yang sebenarnya disembunyikan, lalu mengukur seberapa jauh setiap strategi meleset.

Berkas `P05_nilai_sebenarnya.csv` berisi nilai asli sebelum dikosongkan. Pada proyek Anda berkas semacam ini
tidak akan pernah ada — karena itu jalankan eksperimen ini sekarang, selagi bisa.

In [18]:
# CONTOH 7.1 — eksperimen imputasi
if os.path.exists('P05_nilai_sebenarnya.csv'):
    truth = pd.read_csv('P05_nilai_sebenarnya.csv')
    asli = truth['_saldo_asli']
    obs  = truth['saldo_simpanan']

    hasil = pd.DataFrame({
        'KEBENARAN (semua anggota)' : asli.describe()[['mean','50%','std','max']],
        'Hanya yang terlihat'       : obs.dropna().describe()[['mean','50%','std','max']],
        'Diisi dengan mean'         : obs.fillna(obs.mean()).describe()[['mean','50%','std','max']],
        'Diisi dengan median'       : obs.fillna(obs.median()).describe()[['mean','50%','std','max']],
    }).T.round(0)
    hasil.columns = ['mean', 'median', 'std', 'max']
    display(hasil.map(lambda v: f'{v:,.0f}'))
else:
    print('Berkas kebenaran tidak tersedia; lewati sel ini.')

Berkas kebenaran tidak tersedia; lewati sel ini.


In [19]:
# CONTOH 7.2 — nilai yang disembunyikan itu seperti apa?
if os.path.exists('P05_nilai_sebenarnya.csv'):
    m = obs.isna()
    print('Saldo yang TERLIHAT     — rata-rata:', f'{obs.dropna().mean():>10,.0f}')
    print('Saldo yang DISEMBUNYIKAN— rata-rata:', f'{asli[m].mean():>10,.0f}')
    print()
    print('Rasio:', round(asli[m].mean() / obs.dropna().mean(), 2), 'kali lipat')
    print()
    for nama, seri in [('Buang barisnya', obs.dropna()),
                       ('Isi dengan mean', obs.fillna(obs.mean())),
                       ('Isi dengan median', obs.fillna(obs.median()))]:
        salah = abs(seri.mean() - asli.mean()) / asli.mean() * 100
        print(f'{nama:20s} meleset {salah:5.1f}% dari rata-rata sebenarnya')

> **Baca angka-angka itu pelan-pelan.**
>
> Saldo yang disembunyikan rata-ratanya sekitar **2,8 kali** saldo yang terlihat. Saldo tertinggi yang
> terlihat sekitar 3,5 juta, padahal yang sebenarnya ada mencapai 7,2 juta.
>
> Dan yang paling penting: **ketiga strategi sama-sama meleset 15 sampai 18%.** Membuang baris
> tidak lebih aman daripada mengisi. Keduanya sama-sama menghapus bagian atas sebaran.
>
> Pada data MNAR, tidak ada teknik statistik yang bisa menyelamatkan Anda. Yang bisa dilakukan hanya dua:
> **kembali ke sumber data untuk melengkapi**, atau **melaporkan keterbatasannya secara terbuka** —
> misalnya menulis "rata-rata saldo anggota minimal Rp336.000; angka sebenarnya lebih tinggi karena
> anggota bersaldo besar tidak menampilkan saldonya".
>
> Kalimat seperti itu, di laporan Anda, lebih bernilai daripada angka mana pun.

In [41]:
# CONTOH 7.3 — keputusan kita untuk dataset ini
# saldo_simpanan: TIDAK diimputasi. Dibiarkan kosong dan keterbatasannya dilaporkan.
catat('saldo_simpanan dibiarkan kosong', d['saldo_simpanan'].isna().sum(),
      'Diduga MNAR; imputasi apa pun akan membuat rata-rata saldo tampak lebih kecil daripada kenyataan')

# berat_kg: MAR, dan kekosongannya BERMAKNA (barang non-fisik memang tak punya berat).
d['berlaku_berat'] = ~d['kategori'].isin(['Pulsa', 'Jasa'])
catat('Tandai keberlakuan berat_kg', (~d['berlaku_berat']).sum(),
      'Kosong karena tidak berlaku, bukan karena hilang; diberi kolom penanda, tidak diisi')

# catatan_kasir: kekosongan itu sendiri sebuah kategori.
d['ada_catatan'] = d['catatan_kasir'].notna()
catat('Tandai keberadaan catatan kasir', (~d['ada_catatan']).sum(),
      'Kosong berarti kasir tidak menulis apa pun; dijadikan penanda biner')

### TUGAS 2 — hasil pembersihan

In [66]:
# TUGAS 2
d = d.reset_index(drop=True)

p5_baris_bersih = len(d)
p5_dibuang = len(mentah) - len(d)
p5_rating_kosong = int(d['rating_layanan'].isna().sum())

print("Baris bersih:", p5_baris_bersih)
print("Baris dibuang:", p5_dibuang)
print("Rating kosong:", p5_rating_kosong)


Baris bersih: 1493
Baris dibuang: 50
Rating kosong: 46


In [61]:

print(d.columns.tolist())


['id_transaksi', 'tanggal_transaksi', 'id_anggota', 'nama_produk', 'kategori', 'satuan', 'jumlah', 'harga_satuan', 'diskon_persen', 'total_bayar', 'metode_bayar', 'kasir', 'saldo_simpanan', 'berat_kg', 'rating_layanan', 'catatan_kasir', 'berlaku_berat', 'ada_catatan', 'bulan', 'hari', 'jam', 'nilai_bersih', 'kelas_nilai']


---
## Bagian 8 — Kolom turunan dan encoding

In [43]:
# CONTOH 8.1 — kolom turunan
d['bulan']        = d['tanggal_transaksi'].dt.to_period('M').astype(str)
d['hari']         = d['tanggal_transaksi'].dt.day_name()
d['jam']          = d['tanggal_transaksi'].dt.hour
d['nilai_bersih'] = d['total_bayar']

d[['tanggal_transaksi', 'bulan', 'hari', 'jam']].head(3)

,tanggal_transaksi,bulan,hari,jam
0,2025-08-27 09:43:00,2025-08,Wednesday,9
1,2025-10-09 16:57:00,2025-10,Thursday,16
2,2025-08-27 09:20:00,2025-08,Wednesday,9


In [44]:
# CONTOH 8.2 — binning: angka menjadi kelompok
d['kelas_nilai'] = pd.cut(d['total_bayar'],
                          bins=[0, 10_000, 30_000, 75_000, np.inf],
                          labels=['Sangat kecil', 'Kecil', 'Sedang', 'Besar'])
print(d['kelas_nilai'].value_counts().sort_index())

kelas_nilai
Sangat kecil    299
Kecil           620
Sedang          511
Besar            63
Name: count, dtype: int64


> **Batas bin adalah keputusan, bukan perhitungan.** Angka 10.000, 30.000, dan 75.000 di atas berasal dari
> pengetahuan tentang harga barang koperasi, bukan dari rumus. Tuliskan alasannya di *cleaning log*, karena
> mengubah batas bin akan mengubah kesimpulan analisis Anda.

In [45]:
# CONTOH 8.3 — encoding, dan kapan masing-masing dipakai
# Ordinal: ada urutan, jadi pakai kategori berurutan
urut = ['Sangat kecil', 'Kecil', 'Sedang', 'Besar']
d['kelas_nilai'] = pd.Categorical(d['kelas_nilai'], categories=urut, ordered=True)

# Nominal: tidak ada urutan, jadi one-hot bila nanti dibutuhkan
contoh_onehot = pd.get_dummies(d['metode_bayar'], prefix='bayar', dummy_na=True)
print(contoh_onehot.columns.tolist())
print()
print('Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.')

['bayar_QRIS', 'bayar_Saldo Koperasi', 'bayar_Transfer', 'bayar_Tunai', 'bayar_<NA>']

Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.


> **Jangan melakukan encoding hanya karena bisa.** Pada EDA, kategori lebih mudah dibaca daripada kolom
> nol-satu. One-hot baru berguna saat masuk pemodelan, yang bukan wilayah mata kuliah ini.

---
## Bagian 9 — Menyusun ulang bentuk tabel

**Tidy data** berarti: satu baris satu observasi, satu kolom satu variabel, satu tabel satu satuan amatan.

In [46]:
# CONTOH 9.1 — pivot: dari panjang ke lebar (untuk dibaca manusia)
lebar = (d.pivot_table(index='bulan', columns='kategori', values='total_bayar',
                       aggfunc='sum', observed=True)
          .fillna(0).round(0).astype(int))
lebar

kategori,ATK,Jasa,Makanan,Minuman,Pulsa
bulan,,,,,
2025-08,1498800,901450,5904300,3410675,1828700
2025-09,1251250,824200,5461800,3289175,1417500
2025-10,1337150,1299400,5087800,2817275,1583250
2025-11,1005100,753950,5648750,2571100,1253200


In [47]:
# CONTOH 9.2 — melt: kembali ke panjang (untuk dianalisis mesin)
panjang = lebar.reset_index().melt(id_vars='bulan', var_name='kategori', value_name='total')
print(panjang.shape)
panjang.head(6)

(20, 3)


,bulan,kategori,total
0,2025-08,ATK,1498800
1,2025-09,ATK,1251250
2,2025-10,ATK,1337150
3,2025-11,ATK,1005100
4,2025-08,Jasa,901450
5,2025-09,Jasa,824200


> **Kapan memakai yang mana.** Bentuk **lebar** enak dibaca manusia dan cocok untuk tabel di laporan.
> Bentuk **panjang** adalah yang dibutuhkan hampir semua fungsi grafik dan agregasi. Simpan data Anda
> dalam bentuk panjang, lalu ubah ke lebar hanya pada saat menampilkan.

In [48]:
# CONTOH 9.3 — merge: menggabungkan tabel lain
# Tabel ringkasan per anggota, lalu digabungkan kembali ke data transaksi.
profil_anggota = (d.groupby('id_anggota')
                    .agg(jumlah_transaksi=('id_transaksi', 'count'),
                         total_belanja=('total_bayar', 'sum'))
                    .reset_index())

gabung = d.merge(profil_anggota, on='id_anggota', how='left', validate='many_to_one')
print('Sebelum:', d.shape, '-> sesudah:', gabung.shape)
profil_anggota.head(3)

Sebelum: (1493, 23) -> sesudah: (1493, 25)


,id_anggota,jumlah_transaksi,total_belanja
0,AGT1001,6,190000
1,AGT1002,8,258600
2,AGT1003,5,108000


> **`validate='many_to_one'` itu rem darurat.** Tanpa itu, kunci yang ternyata tidak unik akan membuat
> baris berlipat ganda tanpa peringatan, dan total belanja Anda membengkak. Selalu nyatakan hubungan yang
> Anda harapkan; biarkan pandas yang menolak bila ternyata keliru.
>
> Periksa juga jumlah baris sebelum dan sesudah `merge`. Kalau bertambah, ada yang salah.

---
## Bagian 10 — Pipeline yang dapat dijalankan ulang

Sel-sel yang berserakan tidak bisa dijalankan ulang dengan hasil yang sama. Satukan menjadi satu fungsi.

In [91]:

# CONTOH 10.1 — Pipeline pembersihan data

def bersihkan(df_mentah):
    """Membersihkan data mentah dan membuat log keputusan."""
    x = df_mentah.copy()
    log = []

    def c(langkah, n, alasan):
        log.append({
            'langkah': langkah,
            'baris_terdampak': int(n),
            'alasan': alasan
        })

    # 1. Perbaiki tipe data
    berformat = x['total_bayar'].astype(str).str.startswith('Rp')
    x['total_bayar'] = pd.to_numeric(
        x['total_bayar'].astype(str)
        .str.replace('Rp', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.strip(),
        errors='coerce'
    )
    c('Konversi total_bayar ke angka', berformat.sum(),
      'Menghapus Rp dan pemisah ribuan')

    t = x['tanggal_transaksi'].astype(str)
    iso = t.str.match(r'^\d{4}-')
    tanggal = pd.Series(pd.NaT, index=x.index, dtype='datetime64[ns]')

    tanggal.loc[iso] = pd.to_datetime(
        t.loc[iso], format='%Y-%m-%d %H:%M:%S', errors='coerce'
    )
    tanggal.loc[~iso] = pd.to_datetime(
        t.loc[~iso], format='%d/%m/%Y %H:%M', errors='coerce'
    )
    x['tanggal_transaksi'] = tanggal
    c('Konversi tanggal per format', len(x),
      'Memisahkan format tanggal agar tidak salah dibaca')

    for kol in [
        'jumlah', 'harga_satuan', 'diskon_persen',
        'saldo_simpanan', 'berat_kg', 'rating_layanan'
    ]:
        x[kol] = pd.to_numeric(x[kol], errors='coerce')

    # 2. Buang spasi pada teks
    kolom_teks = x.select_dtypes(
        include=['object', 'string']
    ).columns.tolist()

    n_spasi = 0
    for kol in kolom_teks:
        s = x[kol].astype('string')
        n_spasi += int((s != s.str.strip()).sum())
        x[kol] = s.str.strip()

    c('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Menghindari perbedaan akibat spasi tersembunyi')

    # 3. Ubah placeholder menjadi kosong
    curiga = [
        'n/a', 'na', '-', '', 'null', 'none',
        'tidak diketahui', 'unknown', '?', '#n/a'
    ]
    n_ph = 0

    for kol in kolom_teks:
        mask = x[kol].astype('string').str.lower().isin(curiga)
        n_ph += int(mask.sum())
        x.loc[mask, kol] = pd.NA

    c('Placeholder teks menjadi kosong', n_ph,
      'Placeholder bukan nilai data sebenarnya')

    n999 = int((x['rating_layanan'] == -999).sum())
    x.loc[x['rating_layanan'] == -999, 'rating_layanan'] = np.nan
    c('Placeholder -999 pada rating menjadi kosong', n999,
      'Nilai -999 bukan rating yang valid')

    # 4. Seragamkan kategori
    x['kategori'] = (
        x['kategori'].astype('string').str.strip().str.title()
    )
    peta = {
        'Atk': 'ATK',
        'Alat Tulis': 'ATK',
        'Minumam': 'Minuman'
    }
    sebelum = x['kategori'].nunique()
    x['kategori'] = x['kategori'].replace(peta)

    c('Seragamkan kategori', sebelum - x['kategori'].nunique(),
      'Menyatukan variasi penulisan kategori')

    x['metode_bayar'] = (
        x['metode_bayar'].astype('string')
        .str.strip().str.title()
        .replace({'Qris': 'QRIS'})
    )
    x['kasir'] = (
        x['kasir'].astype('string').str.strip().str.title()
    )

    # 5. Hapus duplikat
    n_persis = int(x.duplicated().sum())
    x = x.drop_duplicates()

    c('Hapus duplikat persis', n_persis,
      'Menyimpan satu baris untuk data identik')

    n_id = int(x['id_transaksi'].duplicated(keep=False).sum())
    x = (
        x.sort_values('catatan_kasir', na_position='last')
        .drop_duplicates(subset=['id_transaksi'], keep='first')
    )

    c('Satukan duplikat menurut id_transaksi', n_id,
      'Menyimpan satu baris untuk setiap ID transaksi')

    n_kosong = int(x['id_transaksi'].isna().sum())
    x = x[x['id_transaksi'].notna()].copy()

    c('Buang baris tanpa id_transaksi', n_kosong,
      'ID transaksi diperlukan untuk penelusuran data')

    # 6. Tangani nilai mustahil
    n_j = int((x['jumlah'] <= 0).sum())
    x.loc[x['jumlah'] <= 0, 'jumlah'] = np.nan

    c('jumlah <= 0 menjadi kosong', n_j,
      'Nilai perlu diperiksa karena bisa berupa kesalahan atau retur')

    buruk = (x['diskon_persen'] < 0) | (x['diskon_persen'] > 100)
    n_dk = int(buruk.sum())
    x.loc[buruk, 'diskon_persen'] = np.nan

    c('Diskon di luar 0-100 menjadi kosong', n_dk,
      'Diskon di luar rentang valid perlu dikonfirmasi')

    x['total_seharusnya'] = (
        x['jumlah'] * x['harga_satuan']
        * (1 - x['diskon_persen'] / 100)
    ).round()

    x['selisih_total'] = x['total_bayar'] - x['total_seharusnya']
    x['total_mencurigakan'] = x['selisih_total'].abs() > 1

    c('Tandai total yang tidak cocok',
      x['total_mencurigakan'].sum(),
      'Menandai selisih tanpa mengubah total asli')

    # 7. Tangani nilai kosong
    c('saldo_simpanan dibiarkan kosong',
      x['saldo_simpanan'].isna().sum(),
      'Tidak melakukan imputasi karena pola kehilangan perlu diperiksa')

    x['berlaku_berat'] = ~x['kategori'].isin(['Pulsa', 'Jasa'])

    c('Tandai keberlakuan berat_kg',
      (~x['berlaku_berat']).sum(),
      'Berat tidak berlaku untuk kategori jasa dan pulsa')

    x['ada_catatan'] = x['catatan_kasir'].notna()

    c('Tandai keberadaan catatan kasir',
      (~x['ada_catatan']).sum(),
      'Menandai transaksi yang tidak memiliki catatan kasir')

    # 8. Buat kolom turunan
    x['bulan'] = x['tanggal_transaksi'].dt.to_period('M').astype(str)
    x['hari'] = x['tanggal_transaksi'].dt.day_name()
    x['jam'] = x['tanggal_transaksi'].dt.hour
    x['nilai_bersih'] = x['total_bayar']

    x['kelas_nilai'] = pd.cut(
        x['total_bayar'],
        bins=[0, 10000, 30000, 75000, np.inf],
        labels=['Sangat kecil', 'Kecil', 'Sedang', 'Besar']
    )

    x['kelas_nilai'] = pd.Categorical(
        x['kelas_nilai'],
        categories=['Sangat kecil', 'Kecil', 'Sedang', 'Besar'],
        ordered=True
    )

    return x.reset_index(drop=True), pd.DataFrame(log)


print("Fungsi bersihkan() berhasil dibuat.")


Fungsi bersihkan() berhasil dibuat.


In [92]:
# Jalankan pipeline pembersihan
d, log_bersih = bersihkan(mentah)

# Tampilkan hasil
print("Ukuran data mentah :", mentah.shape)
print("Ukuran data bersih :", d.shape)
print("Jumlah duplikat    :", d.duplicated().sum())
print("Total nilai kosong :", d.isna().sum().sum())

display(log_bersih)
display(d.head())

Ukuran data mentah : (1543, 16)
Ukuran data bersih : (1493, 26)
Jumlah duplikat    : 0
Total nilai kosong : 1685


,langkah,baris_terdampak,alasan
0,Konversi total_bayar ke angka,250,Menghapus Rp dan pemisah ribuan
1,Konversi tanggal per format,1543,Memisahkan format tanggal agar tidak salah dibaca
2,Buang spasi awal dan akhir kolom teks,160,Menghindari perbedaan akibat spasi tersembunyi
3,Placeholder teks menjadi kosong,19,Placeholder bukan nilai data sebenarnya
4,Placeholder -999 pada rating menjadi kosong,49,Nilai -999 bukan rating yang valid
5,Seragamkan kategori,2,Menyatukan variasi penulisan kategori
6,Hapus duplikat persis,39,Menyimpan satu baris untuk data identik
7,Satukan duplikat menurut id_transaksi,15,Menyimpan satu baris untuk setiap ID transaksi
8,Buang baris tanpa id_transaksi,1,ID transaksi diperlukan untuk penelusuran data
9,jumlah <= 0 menjadi kosong,29,Nilai perlu diperiksa karena bisa berupa kesal...


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir,total_seharusnya,selisih_total,total_mencurigakan,berlaku_berat,ada_catatan,bulan,hari,jam,nilai_bersih,kelas_nilai
0,TRX250665,2025-08-27 09:43:00,AGT1014,Map Plastik,ATK,pcs,5.0,4000,0.0,20000,Tunai,Dimas,45000.0,1.720,5.0,Minta struk,20000.0,0.0,False,True,True,2025-08,Wednesday,9,20000,Kecil
1,TRX251255,2025-10-09 16:57:00,AGT1148,Map Plastik,ATK,pcs,NaN,4000,0.0,12000,<NA>,Bagus,159000.0,3.589,5.0,Minta struk,NaN,NaN,False,True,True,2025-10,Thursday,16,12000,Kecil
2,TRX250482,2025-08-27 09:20:00,AGT1102,Gorengan 5pcs,Makanan,porsi,4.0,6000,0.0,24000,Tunai,Bagus,NaN,2.008,NaN,Minta struk,24000.0,0.0,False,True,True,2025-08,Wednesday,9,24000,Kecil
3,TRX250414,2025-10-30 07:58:00,AGT1081,Pulpen Hitam,ATK,pcs,6.0,3500,0.0,21000,QRIS,Rina,NaN,3.598,5.0,Minta struk,21000.0,0.0,False,True,True,2025-10,Thursday,7,21000,Kecil
4,TRX250097,2025-09-06 17:47:00,AGT1053,Air Mineral 600ml,Minuman,botol,8.0,4000,0.0,32000,Tunai,Rina,35000.0,3.433,4.0,Minta struk,32000.0,0.0,False,True,True,2025-09,Saturday,17,32000,Sedang


> **Uji reproduktibilitas yang murah:** jalankan `bersihkan(mentah)` dua kali, lalu bandingkan hasilnya.
> Kalau berbeda, ada yang bergantung pada urutan acak atau pada keadaan sel sebelumnya. Pada pertemuan 15
> tim lain akan menjalankan ulang notebook Anda, jadi masalah ini lebih murah diselesaikan sekarang.

In [93]:

# CONTOH 10.2 — Simpan hasil

import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

d.to_csv('data/processed/transaksi_bersih.csv', index=False)

print('Tersimpan:', d.shape)
print('Data mentah tidak disentuh:', mentah.shape)


Tersimpan: (1493, 26)
Data mentah tidak disentuh: (1543, 16)


---
## Bagian 11 — Cleaning log

Dokumen ini yang membedakan pembersihan yang bisa dipertanggungjawabkan dari pembersihan yang asal jalan.
Setiap baris menjawab tiga hal: **apa yang diubah, berapa banyak, dan mengapa**.

In [94]:

# BAGIAN 11 — Simpan cleaning_log.md

log = log_bersih.copy()

baris = [
    '# Cleaning Log',
    '',
    'Dataset: transaksi_koperasi_2025.csv  ',
    f'Baris mentah: {len(mentah)} -> baris bersih: {len(d)}  ',
    '',
    '| # | Langkah | Baris terdampak | Alasan |',
    '|---|---|---|---|'
]

for i, r in log.iterrows():
    baris.append(
        f"| {i + 1} | {r['langkah']} | "
        f"{r['baris_terdampak']} | {r['alasan']} |"
    )

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('Cleaning log berhasil disimpan!')
print('Jumlah langkah:', len(log))
print('Lokasi: cleaning_log.md')


Cleaning log berhasil disimpan!
Jumlah langkah: 15
Lokasi: cleaning_log.md


In [52]:
# CONTOH 11.2 — simpan sebagai Markdown untuk repositori
baris = ['# Cleaning Log', '',
         'Dataset: transaksi_koperasi_2025.csv  ',
         f'Baris mentah: {len(mentah)} -> baris bersih: {len(d)}  ',
         '', '| # | Langkah | Baris terdampak | Alasan |', '|---|---|---|---|']
for i, r in log.iterrows():
    baris.append(f"| {i} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('\n'.join(baris[:10]))

# Cleaning Log

Dataset: transaksi_koperasi_2025.csv  
Baris mentah: 1543 -> baris bersih: 1493  

| # | Langkah | Baris terdampak | Alasan |
|---|---|---|---|
| 1 | Konversi total_bayar ke angka | 250 | Menghapus Rp dan pemisah ribuan agar nilai dapat dihitung. |
| 2 | Konversi total_bayar ke angka | 0 | Buang Rp dan pemisah ribuan agar kolom dapat dihitung |
| 3 | Konversi tanggal per format | 1543 | format="mixed" dengan dayfirst merusak 483 baris berformat ISO tanpa memunculkan galat |


### TUGAS 3 — kelengkapan log

In [73]:

# TUGAS 3

p5_langkah = len(jejak)
p5_total_num = int(d['total_bayar'].notna().sum())
p5_bulan = d['bulan'].nunique(dropna=True)

cek('P5_langkah', p5_langkah)
cek('P5_total_num', p5_total_num)
cek('P5_bulan', p5_bulan)


BELUM P5_langkah = 21
Petunjuk: Jumlah langkah di cleaning log.
BENAR P5_total_num = 1493
BENAR P5_bulan = 4


In [75]:
import os

os.makedirs('reports', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)

print("Folder reports dan notebooks berhasil dibuat!")

Folder reports dan notebooks berhasil dibuat!


In [80]:
import os

path = '/content/drive/MyDrive/exploratory data analysis/p5/TEMPLATE_Data_Quality_Report (1).docx'

print(os.path.exists(path))

True


In [82]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00


In [83]:
from docx import Document

doc = Document(path)

for paragraf in doc.paragraphs:
    print(paragraf.text)

TEMPLATE — M2: DATA QUALITY REPORT
Exploratory Data Analysis (KMTI21133) · Bobot 10% · Dikumpulkan pada pertemuan 5
Hapus seluruh teks miring (petunjuk) sebelum mengumpulkan.
1. Ringkasan
Lima sampai delapan kalimat. Ditulis paling akhir, dibaca paling awal. Sebutkan: dataset apa, berapa besar, tiga masalah terbesar yang ditemukan, keputusan paling berpengaruh yang diambil, dan apakah data ini layak dianalisis. Pembaca yang hanya membaca bagian ini harus tahu kondisi data Anda.





2. Data dan cara memeriksanya
2.1 Asal data
Salin lima butir provenans dari README: nama sumber, tautan atau instansi, tanggal pengambilan, lisensi atau nomor surat izin, dan kalimat sitasi.
2.2 Ukuran dan cakupan
Jumlah baris dan kolom, satuan amatan (satu baris mewakili apa), serta rentang waktu yang tercakup.



2.3 Cara pemeriksaan
Satu paragraf: pemeriksaan apa saja yang dijalankan dan dengan alat apa. Sebutkan bila ada pemeriksaan yang belum bisa dilakukan beserta alasannya.




3. Temuan kualitas dat

In [84]:
from docx import Document

doc = Document(path)

print("=== ISI TEMPLATE ===")
for p in doc.paragraphs:
    if p.text.strip():
        print(p.text)

print("\n=== TABEL TEMPLATE ===")
for i, tabel in enumerate(doc.tables):
    print(f"\nTabel {i+1}:")
    for baris in tabel.rows:
        print([sel.text for sel in baris.cells])

=== ISI TEMPLATE ===
TEMPLATE — M2: DATA QUALITY REPORT
Exploratory Data Analysis (KMTI21133) · Bobot 10% · Dikumpulkan pada pertemuan 5
Hapus seluruh teks miring (petunjuk) sebelum mengumpulkan.
1. Ringkasan
Lima sampai delapan kalimat. Ditulis paling akhir, dibaca paling awal. Sebutkan: dataset apa, berapa besar, tiga masalah terbesar yang ditemukan, keputusan paling berpengaruh yang diambil, dan apakah data ini layak dianalisis. Pembaca yang hanya membaca bagian ini harus tahu kondisi data Anda.
2. Data dan cara memeriksanya
2.1 Asal data
Salin lima butir provenans dari README: nama sumber, tautan atau instansi, tanggal pengambilan, lisensi atau nomor surat izin, dan kalimat sitasi.
2.2 Ukuran dan cakupan
Jumlah baris dan kolom, satuan amatan (satu baris mewakili apa), serta rentang waktu yang tercakup.
2.3 Cara pemeriksaan
Satu paragraf: pemeriksaan apa saja yang dijalankan dan dengan alat apa. Sebutkan bila ada pemeriksaan yang belum bisa dilakukan beserta alasannya.
3. Temuan kua

In [85]:
import os

folder = '/content/drive/MyDrive/exploratory data analysis/p5'

for nama in os.listdir(folder):
    print(nama)

P05_Workbook_Mandiri.ipynb
P05_nilai_sebenarnya.csv
TEMPLATE_cleaning_log (1).md
TEMPLATE_Data_Quality_Report (1).docx
transaksi_koperasi_2025 (1).csv


In [86]:
print("Data mentah:", mentah.shape)
print("Data bersih:", d.shape)

print("\nKolom data mentah:")
print(mentah.columns.tolist())

print("\nKolom data bersih:")
print(d.columns.tolist())

Data mentah: (1543, 16)
Data bersih: (1493, 26)

Kolom data mentah:
['id_transaksi', 'tanggal_transaksi', 'id_anggota', 'nama_produk', 'kategori', 'satuan', 'jumlah', 'harga_satuan', 'diskon_persen', 'total_bayar', 'metode_bayar', 'kasir', 'saldo_simpanan', 'berat_kg', 'rating_layanan', 'catatan_kasir']

Kolom data bersih:
['id_transaksi', 'tanggal_transaksi', 'id_anggota', 'nama_produk', 'kategori', 'satuan', 'jumlah', 'harga_satuan', 'diskon_persen', 'total_bayar', 'metode_bayar', 'kasir', 'saldo_simpanan', 'berat_kg', 'rating_layanan', 'catatan_kasir', 'berlaku_berat', 'ada_catatan', 'bulan', 'hari', 'jam', 'nilai_bersih', 'kelas_nilai', 'total_seharusnya', 'selisih_total', 'total_mencurigakan']


In [87]:
print("=== DATA MENTAH ===")
print("Sel kosong:", mentah.isna().sum().sum())
print("Baris duplikat:", mentah.duplicated().sum())

print("\n=== DATA BERSIH ===")
print("Sel kosong:", d.isna().sum().sum())
print("Baris duplikat:", d.duplicated().sum())

print("\n=== KOLOM YANG MASIH KOSONG ===")
kosong = d.isna().sum()
print(kosong[kosong > 0].sort_values(ascending=False))

print("\n=== KATEGORI ===")
print("Kategori:", d["kategori"].nunique())
print("Metode bayar:", d["metode_bayar"].nunique())
print("Anggota:", d["id_anggota"].nunique())

=== DATA MENTAH ===
Sel kosong: 1570
Baris duplikat: 24

=== DATA BERSIH ===
Sel kosong: 1685
Baris duplikat: 0

=== KOLOM YANG MASIH KOSONG ===
catatan_kasir       1004
berat_kg             341
saldo_simpanan       156
rating_layanan        46
total_seharusnya      36
selisih_total         36
jumlah                29
metode_bayar          28
diskon_persen          9
dtype: int64

=== KATEGORI ===
Kategori: 5
Metode bayar: 4
Anggota: 259


In [88]:
print("=== BERAT KOSONG BERDASARKAN KATEGORI ===")
print(
    d.assign(kosong=d["berat_kg"].isna())
     .groupby("kategori", dropna=False)["kosong"]
     .agg(["sum", "count", "mean"])
)

print("\n=== SALDO KOSONG BERDASARKAN METODE BAYAR ===")
print(
    d.assign(kosong=d["saldo_simpanan"].isna())
     .groupby("metode_bayar", dropna=False)["kosong"]
     .agg(["sum", "count", "mean"])
)

print("\n=== CATATAN KASIR KOSONG BERDASARKAN KATEGORI ===")
print(
    d.assign(kosong=d["catatan_kasir"].isna())
     .groupby("kategori", dropna=False)["kosong"]
     .agg(["sum", "count", "mean"])
     .round(3)
)

=== BERAT KOSONG BERDASARKAN KATEGORI ===
          sum  count      mean
kategori                      
ATK        51    311  0.163987
Jasa      149    149  1.000000
Makanan     0    488  0.000000
Minuman     0    404  0.000000
Pulsa     141    141  1.000000

=== SALDO KOSONG BERDASARKAN METODE BAYAR ===
                sum  count      mean
metode_bayar                        
QRIS             46    465  0.098925
Saldo Koperasi   35    279  0.125448
Transfer         14    171  0.081871
Tunai            61    550  0.110909
<NA>              0     28  0.000000

=== CATATAN KASIR KOSONG BERDASARKAN KATEGORI ===
          sum  count   mean
kategori                   
ATK       188    311  0.605
Jasa      104    149  0.698
Makanan   338    488  0.693
Minuman   270    404  0.668
Pulsa     104    141  0.738


In [89]:
kolom = "total_bayar"

print("=== SEBELUM DIBERSIHKAN ===")
print(mentah[kolom].describe())

print("\n=== SESUDAH DIBERSIHKAN ===")
print(d[kolom].describe())

=== SEBELUM DIBERSIHKAN ===
count      1543
unique      220
top       12000
freq         79
Name: total_bayar, dtype: object

=== SESUDAH DIBERSIHKAN ===
count      1493.000000
mean      32916.828533
std       47456.414119
min         250.000000
25%       12000.000000
50%       24000.000000
75%       40000.000000
max      800000.000000
Name: total_bayar, dtype: float64


> Kalau jumlah langkah Anda berbeda dari kunci, itu **tidak otomatis salah** — tim boleh memecah atau
> menggabungkan langkah. Yang penting setiap perubahan ada catatannya. Angka pada kunci mengikuti
> pembagian langkah di workbook ini.

---
## Bagian 12 — Verifikasi: jalankan ulang diagnosis pertemuan 4

Pembersihan belum selesai sampai Anda membuktikan bahwa masalahnya benar-benar hilang —
dan bahwa tidak ada masalah baru yang Anda ciptakan sendiri.

In [95]:

# BAGIAN 12.1 — Diagnosis ulang

def diagnosa_singkat(data, nama):
    print(f'=== {nama} ===')
    print('Baris                     :', len(data))
    print('Duplikat persis           :', int(data.duplicated().sum()))
    print('Kategori unik             :', data['kategori'].astype(str).nunique())
    print('ID anggota unik            :', data['id_anggota'].nunique())
    print('total_bayar bertipe angka :',
          pd.api.types.is_numeric_dtype(data['total_bayar']))
    print('Rating bernilai -999      :',
          int((pd.to_numeric(data['rating_layanan'], errors='coerce') == -999).sum()))
    print('jumlah <= 0               :',
          int((pd.to_numeric(data['jumlah'], errors='coerce') <= 0).sum()))
    print()

diagnosa_singkat(mentah, 'SEBELUM')
diagnosa_singkat(d, 'SESUDAH')


# BAGIAN 12.2 — Bandingkan nilai kosong

banding = pd.DataFrame({
    'kosong_sebelum': mentah.isna().sum(),
    'kosong_sesudah': d.reindex(columns=mentah.columns).isna().sum()
})

banding['selisih'] = (
    banding['kosong_sesudah'] - banding['kosong_sebelum']
)

display(banding[banding['selisih'] != 0].sort_values(
    'selisih', ascending=False
))


=== SEBELUM ===
Baris                     : 1543
Duplikat persis           : 24
Kategori unik             : 18
ID anggota unik            : 318
total_bayar bertipe angka : False
Rating bernilai -999      : 49
jumlah <= 0               : 29

=== SESUDAH ===
Baris                     : 1493
Duplikat persis           : 0
Kategori unik             : 5
ID anggota unik            : 259
total_bayar bertipe angka : True
Rating bernilai -999      : 0
jumlah <= 0               : 0



,kosong_sebelum,kosong_sesudah,selisih
rating_layanan,0,46,46
jumlah,0,29,29
metode_bayar,9,28,19
diskon_persen,0,9,9
saldo_simpanan,159,156,-3
id_transaksi,7,0,-7
berat_kg,351,341,-10
catatan_kasir,1044,1004,-40


In [55]:
# CONTOH 12.2 — apa yang HILANG dan apa yang BERTAMBAH
banding = pd.DataFrame({
    'kosong_sebelum': mentah.isna().sum(),
    'kosong_sesudah': d.reindex(columns=mentah.columns).isna().sum(),
})
banding['selisih'] = banding['kosong_sesudah'] - banding['kosong_sebelum']
banding[banding['selisih'] != 0].sort_values('selisih', ascending=False)

,kosong_sebelum,kosong_sesudah,selisih
rating_layanan,0,46,46
jumlah,0,29,29
metode_bayar,9,28,19
diskon_persen,0,9,9
saldo_simpanan,159,156,-3
id_transaksi,7,0,-7
berat_kg,351,341,-10
catatan_kasir,1044,1004,-40


> **Nilai kosong Anda bertambah, dan itu wajar.** Placeholder yang tadinya menyamar sebagai angka kini
> jujur mengaku kosong. Data yang lebih jujur memang terlihat lebih buruk, dan itulah tandanya pembersihan
> Anda berhasil.
>
> Laporkan angka ini apa adanya di M2. Laporan yang mengaku "nilai kosong bertambah dari 1.570 menjadi
> sekian karena placeholder diungkap" jauh lebih meyakinkan daripada laporan yang semuanya nol.

---
## Bagian 13 — Kerjakan untuk proyek tim: M2

**M2 — Data Quality Report, bobot 10%.** Tiga berkas yang dikumpulkan:

| Berkas | Isi | Diperiksa oleh rubrik |
|---|---|---|
| `reports/data_quality_report.docx` | Temuan, keputusan, keterbatasan | Kelengkapan diagnosis (25), Justifikasi (30) |
| `notebooks/02_cleaning.ipynb` | Pipeline yang dapat dijalankan ulang | Ketepatan implementasi (20) |
| `cleaning_log.md` | Setiap langkah, jumlah, dan alasan | Dokumentasi dan keterlacakan (15) |

Dimensi kelima, **Penanganan data pribadi (10)**, dinilai dari apakah data olahan Anda tetap aman —
yaitu lanjutan pekerjaan pertemuan 3.

Gunakan `TEMPLATE_Data_Quality_Report.docx` sebagai kerangka laporan.

In [56]:
# 13.1 — salin pola workbook ini ke dataset tim
# df_tim = pd.read_csv('data/raw/....csv')
# bersih_tim, log_tim = bersihkan(df_tim)
# bersih_tim.to_csv('data/processed/....csv', index=False)


In [57]:
# 13.2 — jalankan pemeriksa kelengkapan sebelum mengumpulkan
# Dari terminal, di dalam folder repositori tim:
#     python cek_m2.py
# Skrip akan memeriksa keberadaan berkas, struktur folder, dan isi minimal cleaning log.


In [96]:
from docx import Document
import os

folder = '/content/drive/MyDrive/exploratory data analysis/p5'
template = os.path.join(folder, 'TEMPLATE_Data_Quality_Report (1).docx')

os.makedirs('reports', exist_ok=True)

doc = Document(template)

print("Template berhasil dibuka.")
print("Isi template:")
for p in doc.paragraphs:
    if p.text.strip():
        print(p.text)


Template berhasil dibuka.
Isi template:
TEMPLATE — M2: DATA QUALITY REPORT
Exploratory Data Analysis (KMTI21133) · Bobot 10% · Dikumpulkan pada pertemuan 5
Hapus seluruh teks miring (petunjuk) sebelum mengumpulkan.
1. Ringkasan
Lima sampai delapan kalimat. Ditulis paling akhir, dibaca paling awal. Sebutkan: dataset apa, berapa besar, tiga masalah terbesar yang ditemukan, keputusan paling berpengaruh yang diambil, dan apakah data ini layak dianalisis. Pembaca yang hanya membaca bagian ini harus tahu kondisi data Anda.
2. Data dan cara memeriksanya
2.1 Asal data
Salin lima butir provenans dari README: nama sumber, tautan atau instansi, tanggal pengambilan, lisensi atau nomor surat izin, dan kalimat sitasi.
2.2 Ukuran dan cakupan
Jumlah baris dan kolom, satuan amatan (satu baris mewakili apa), serta rentang waktu yang tercakup.
2.3 Cara pemeriksaan
Satu paragraf: pemeriksaan apa saja yang dijalankan dan dengan alat apa. Sebutkan bila ada pemeriksaan yang belum bisa dilakukan beserta alasa

In [100]:

from docx import Document
from docx.shared import Pt
import os
import pandas as pd
import numpy as np

# ==========================================
# 1. PERSIAPAN LAPORAN
# ==========================================
folder = '/content/drive/MyDrive/exploratory data analysis/p5'
template = os.path.join(
    folder,
    'TEMPLATE_Data_Quality_Report (1).docx'
)

os.makedirs('reports', exist_ok=True)

doc = Document(template)

# Kosongkan isi template tanpa menghapus pengaturan halaman
body = doc._element.body
for child in list(body):
    if child.tag.endswith('}sectPr'):
        continue
    body.remove(child)


# ==========================================
# 2. FUNGSI BANTU
# ==========================================
def judul(teks, level=1):
    p = doc.add_paragraph()
    r = p.add_run(str(teks))
    r.bold = True
    r.font.size = Pt(16 if level == 1 else 13)


def tulis(teks=''):
    doc.add_paragraph(str(teks))


def buat_tabel(data, headers):
    # Tidak menggunakan style bawaan template
    tabel_doc = doc.add_table(rows=1, cols=len(headers))

    for i, header in enumerate(headers):
        tabel_doc.rows[0].cells[i].text = str(header)

    for baris in data:
        sel = tabel_doc.add_row().cells
        for i, nilai in enumerate(baris):
            sel[i].text = str(nilai)

    return tabel_doc


# ==========================================
# 3. RINGKASAN DATA
# ==========================================
kolom_asli = list(mentah.columns)

jumlah_kosong_awal = int(mentah.isna().sum().sum())

jumlah_kosong_akhir = int(
    d.reindex(columns=kolom_asli).isna().sum().sum()
)

duplikat_awal = int(mentah.duplicated().sum())
duplikat_akhir = int(d.duplicated().sum())

# Konversi total_bayar mentah menjadi angka
total_awal = pd.to_numeric(
    mentah['total_bayar']
    .astype(str)
    .str.replace('Rp', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.strip(),
    errors='coerce'
)

total_akhir = pd.to_numeric(
    d['total_bayar'],
    errors='coerce'
)


# ==========================================
# 4. HALAMAN JUDUL DAN RINGKASAN
# ==========================================
judul('M2 — DATA QUALITY REPORT')
tulis('Exploratory Data Analysis (KMTI21133)')
tulis('Dataset: transaksi_koperasi_2025')

judul('1. Ringkasan', 2)

tulis(
    f'Dataset mentah berisi {len(mentah)} baris dan '
    f'{len(kolom_asli)} kolom. Setelah proses pembersihan, '
    f'dataset memiliki {len(d)} baris dan {len(d.columns)} kolom, '
    'termasuk kolom turunan.'
)

tulis(
    f'Tiga masalah utama adalah nilai kosong, {duplikat_awal} '
    'baris duplikat persis, serta ketidakkonsistenan format '
    'atau validitas beberapa nilai.'
)

tulis(
    f'Pembersihan mengurangi jumlah baris sebanyak '
    f'{len(mentah) - len(d)} baris. Duplikat persis setelah '
    f'pembersihan berjumlah {duplikat_akhir}.'
)

tulis(
    'Data dapat digunakan untuk eksplorasi awal. Namun, nilai kosong, '
    'aturan pembersihan, dan asumsi analisis perlu diperhatikan.'
)


# ==========================================
# 5. DATA DAN METODE
# ==========================================
judul('2. Data dan Metode', 2)

judul('2.1 Provenance', 2)
tulis('Nama sumber: transaksi_koperasi_2025 (CSV).')

tulis(
    'Tautan atau institusi sumber, tanggal pengumpulan, lisensi atau '
    'izin, dan sitasi belum dapat dipastikan dari informasi yang '
    'tersedia. Informasi tersebut perlu dilengkapi dari README '
    'atau keterangan resmi dataset.'
)

judul('2.2 Ukuran dan Cakupan', 2)
tulis(f'Jumlah baris mentah: {len(mentah)}.')
tulis(f'Jumlah kolom asli: {len(kolom_asli)}.')
tulis(f'Jumlah baris setelah pembersihan: {len(d)}.')

if 'tanggal_transaksi' in mentah.columns:
    tanggal = pd.to_datetime(
        mentah['tanggal_transaksi'],
        errors='coerce'
    )
    tulis(
        f'Rentang tanggal transaksi: {tanggal.min()} '
        f'sampai {tanggal.max()}.'
    )

judul('2.3 Metode', 2)
tulis(
    'Pemeriksaan mencakup tipe data, nilai kosong, duplikat, '
    'konsistensi kategori, validitas nilai, dan perbandingan '
    'statistik sebelum serta sesudah pembersihan.'
)


# ==========================================
# 6. TEMUAN KUALITAS DATA
# ==========================================
judul('3. Temuan Kualitas Data', 2)

buat_tabel(
    [
        [
            'Completeness',
            f'{jumlah_kosong_awal} sel kosong pada data mentah',
            'Tinggi'
        ],
        [
            'Uniqueness',
            f'{duplikat_awal} baris duplikat persis',
            'Sedang'
        ],
        [
            'Consistency',
            'Format angka, teks, dan kategori perlu diperiksa',
            'Sedang'
        ],
        [
            'Validity',
            'Nilai yang tidak sesuai aturan perlu diperiksa',
            'Sedang'
        ],
        [
            'Accuracy',
            'Belum diverifikasi terhadap sumber acuan',
            'Belum diketahui'
        ]
    ],
    ['Aspek', 'Temuan', 'Tingkat']
)

judul('3.1 Pemeriksaan Nilai Kosong', 2)

tulis(
    'Tabel berikut menunjukkan kolom dengan nilai kosong lebih '
    'dari 5% pada data mentah. Penyebab nilai kosong perlu '
    'diinvestigasi sebelum menentukan apakah termasuk MCAR, '
    'MAR, atau MNAR.'
)

data_missing = []

for kolom in kolom_asli:
    jumlah = int(mentah[kolom].isna().sum())
    persen = jumlah / len(mentah) * 100

    if persen > 5:
        data_missing.append([
            kolom,
            jumlah,
            f'{persen:.2f}%',
            'Penyebab perlu diperiksa'
        ])

if data_missing:
    buat_tabel(
        data_missing,
        ['Kolom', 'Jumlah kosong', 'Persentase', 'Catatan']
    )
else:
    tulis('Tidak ada kolom dengan nilai kosong di atas 5%.')

tulis(
    'Nilai kosong pada catatan_kasir tidak otomatis diisi karena '
    'catatan mungkin memang tidak dibuat. Nilai kosong pada '
    'saldo_simpanan tidak diimputasi karena penyebabnya belum '
    'diketahui. Nilai berat_kg pada kategori Jasa dan Pulsa perlu '
    'dikonfirmasi apakah memang tidak relevan.'
)


# ==========================================
# 7. KEPUTUSAN PEMBERSIHAN
# ==========================================
judul('4. Keputusan Pembersihan dan Alasan', 2)

buat_tabel(
    [
        [
            'Konversi total_bayar',
            'Mengubah nilai mata uang menjadi angka untuk analisis.'
        ],
        [
            'Penyeragaman teks dan spasi',
            'Mengurangi variasi penulisan yang tidak bermakna.'
        ],
        [
            'Penanganan placeholder',
            'Menandai teks penanda data hilang sebagai nilai kosong.'
        ],
        [
            'Penyeragaman kategori',
            'Membuat penulisan kategori lebih konsisten.'
        ],
        [
            'Penghapusan duplikat persis',
            'Menghindari penghitungan baris identik berulang kali.'
        ],
        [
            'Pemeriksaan nilai tidak valid',
            'Menangani nilai yang tidak sesuai aturan validitas.'
        ],
        [
            'Tidak mengisi semua nilai kosong dengan nol',
            'Alternatif ini ditolak karena nol dapat mengubah makna data.'
        ],
        [
            'Penambahan kolom turunan',
            'Mendukung pemeriksaan dan analisis lanjutan.'
        ]
    ],
    ['Keputusan', 'Alasan']
)


# ==========================================
# 8. DAMPAK PEMBERSIHAN
# ==========================================
judul('5. Dampak Pembersihan', 2)

tulis(f'Sel kosong sebelum pembersihan: {jumlah_kosong_awal}.')
tulis(
    'Sel kosong sesudah pembersihan pada kolom asli: '
    f'{jumlah_kosong_akhir}.'
)
tulis(f'Duplikat sebelum pembersihan: {duplikat_awal}.')
tulis(f'Duplikat sesudah pembersihan: {duplikat_akhir}.')
tulis(
    f'Baris yang berkurang: {len(mentah) - len(d)}.'
)

data_statistik = []

for nama, seri in [
    ('Sebelum', total_awal),
    ('Sesudah', total_akhir)
]:
    data_statistik.append([
        nama,
        round(seri.mean(), 2),
        round(seri.median(), 2),
        round(seri.std(), 2),
        int(seri.notna().sum())
    ])

buat_tabel(
    data_statistik,
    [
        'Data',
        'Rata-rata total_bayar',
        'Median',
        'Standar deviasi',
        'Jumlah angka'
    ]
)


# ==========================================
# 9. KETERBATASAN
# ==========================================
judul('6. Keterbatasan', 2)

tulis(
    'Informasi sumber dataset belum lengkap. Kebenaran transaksi '
    'belum dapat diverifikasi terhadap sumber eksternal. Pola nilai '
    'kosong belum cukup untuk memastikan mekanisme MCAR, MAR, atau '
    'MNAR. Kolom turunan juga bergantung pada aturan yang digunakan '
    'dalam proses pembersihan.'
)


# ==========================================
# 10. PENANGANAN DATA PRIBADI
# ==========================================
judul('7. Penanganan Data Pribadi', 2)

tulis(
    'Kolom id_anggota, nama, dan catatan yang berpotensi '
    'mengidentifikasi individu perlu dibatasi aksesnya. Untuk laporan '
    'yang dibagikan, hindari menampilkan identitas anggota secara '
    'langsung dan gunakan data agregat jika memungkinkan.'
)


# ==========================================
# 11. LANGKAH MENUJU M3
# ==========================================
judul('8. Langkah Berikutnya Menuju M3', 2)

tulis(
    'Lengkapi informasi sumber dataset, konfirmasi arti nilai kosong '
    'pada berat_kg dan saldo_simpanan, periksa transaksi mencurigakan, '
    'dan dokumentasikan asumsi. Selanjutnya lakukan EDA melalui '
    'visualisasi distribusi, perbandingan kategori, dan tren waktu.'
)


# ==========================================
# 12. LAMPIRAN A — CLEANING LOG
# ==========================================
judul('Lampiran A — Cleaning Log', 2)

if 'log_bersih' in globals() and not log_bersih.empty:
    kolom_log = [
        kolom for kolom in [
            'langkah',
            'baris_terdampak',
            'alasan'
        ]
        if kolom in log_bersih.columns
    ]

    if kolom_log:
        buat_tabel(
            log_bersih[kolom_log].values.tolist(),
            kolom_log
        )
    else:
        tulis('Kolom cleaning log perlu diperiksa.')
else:
    tulis(
        'Cleaning log belum tersedia pada sesi ini. '
        'Jalankan pipeline pembersihan untuk melengkapinya.'
    )


# ==========================================
# 13. LAMPIRAN B — MANUAL MAPPING
# ==========================================
judul('Lampiran B — Manual Mapping', 2)

tulis(
    'Mapping manual perlu dilengkapi sesuai aturan yang benar-benar '
    'digunakan dalam notebook. Mapping tambahan tidak diasumsikan '
    'dalam laporan ini.'
)


# ==========================================
# 14. SIMPAN LAPORAN
# ==========================================
output = 'reports/data_quality_report.docx'
doc.save(output)

print('BERHASIL! Laporan tersimpan di:', output)


BERHASIL! Laporan tersimpan di: reports/data_quality_report.docx


### Daftar periksa M2

- [ ] Seluruh temuan pertemuan 4 muncul di laporan, termasuk yang Anda putuskan untuk dibiarkan
- [ ] Setiap keputusan punya alasan tertulis, bukan sekadar deskripsi tindakan
- [ ] Minimal satu keputusan menyebutkan **alternatif yang dipertimbangkan dan ditolak**
- [ ] Dampak terhadap sebaran data ditunjukkan untuk minimal satu kolom (pola Bagian 7)
- [ ] Keterbatasan dinyatakan terbuka, termasuk kolom yang tidak bisa diselamatkan
- [ ] `data/raw/` tidak berubah sama sekali dan tidak ikut diunggah
- [ ] Notebook berjalan dari awal sampai akhir tanpa galat setelah *Restart and run all*
- [ ] `cleaning_log.md` ada di akar repositori
- [ ] `python cek_m2.py` berjalan tanpa keluhan

> **Kesalahan yang paling mahal pada M2:** menulis apa yang dilakukan, bukan mengapa.
> "Kolom X diisi dengan median" adalah deskripsi. "Kolom X diisi dengan median karena kekosongannya
> tersebar merata di semua kelompok dan hanya 3%, sehingga pilihan ini tidak menggeser sebaran lebih dari
> 1%; alternatif membuang baris ditolak karena akan menghapus 48 transaksi yang kolom lainnya lengkap"
> adalah justifikasi. Rubrik memberi bobot 30 untuk yang kedua.

---
## Kalau Anda mentok

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca baris terakhir pesan galat |
| 2 | *Runtime → Restart and run all*. Banyak galat muncul karena sel dijalankan tidak berurutan. |
| 3 | Cetak `d.dtypes` dan `d.shape` sebelum serta sesudah langkah yang bermasalah |
| 4 | Tanyakan ke anggota tim |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di forum tanya-jawab LMS; dosen menjawab setiap hari kerja |

**Galat yang paling sering muncul di pertemuan ini:**

| Pesan | Artinya | Perbaikannya |
|---|---|---|
| `can only use .str accessor with string values` | Kolomnya bukan teks | `.astype(str)` atau `.astype('string')` lebih dulu |
| `SettingWithCopyWarning` | Menulis ke salinan, bukan ke tabel aslinya | Pakai `.loc[baris, kolom] = nilai`, dan `.copy()` setelah menyaring |
| `could not convert string to float` | Masih ada teks di kolom angka | Pakai `errors='coerce'`, lalu hitung berapa yang gagal |
| `MergeError: Merge keys are not unique` | Kunci gabungan tidak unik | Itu justru gunanya `validate=`; perbaiki kuncinya, jangan hapus validasinya |

---
## Ringkasan pertemuan 5

1. **Pembersihan adalah rangkaian keputusan.** Yang dinilai adalah alasannya, bukan pilihannya.
2. **Urutan menentukan hasil.** Tipe data dulu, nilai kosong paling akhir.
3. **Menandai lebih aman daripada menebak.** Kolom penanda memindahkan keputusan ke tahap yang tepat.
4. **Pada data MNAR, semua strategi meleset** — termasuk membuang baris. Yang menolong hanya kembali ke
   sumber data, atau melaporkan keterbatasannya secara terbuka.
5. **Nilai kosong yang bertambah adalah tanda keberhasilan**, bukan kegagalan.
6. **Pembersihan belum selesai sampai diagnosisnya dijalankan ulang.**

### Setelah pertemuan 5

- Selesaikan dan kumpulkan **M2** sesuai tenggat di LMS.
- Kerjakan kuis formatif pertemuan 5.
- Pertemuan 6 kembali bersama dosen: analisis univariat, *outlier*, dan visualisasi yang jujur.